# Entity extraction with BIO tags

A small import business wants a program that copies the supplier, the date and the amount out of the free-text
notes on its bank transfers. In this notebook you tag the lesson's six notes with BIO tags, align the word tags to
the tokenizer's pieces with `word_ids()`, fine-tune a small BERT on 600 generated notes (once with the correct
alignment, once with the by-position bug), and run Hugging Face's token-classification pipeline with
`aggregation_strategy` "none", "simple" and "first".

About 10 minutes. Run the cells from top to bottom. Everything runs on a free CPU runtime, with no account and no
key. The model, `google/bert_uncased_L-4_H-256_A-4` (a 4-layer BERT, about 45 MB), is downloaded from the
Hugging Face Hub. A warning about `HF_TOKEN` is normal; you do not need one.

## 1. Install the transformers version of the lesson

Colab already has PyTorch and transformers. The next cell installs the transformers version the lesson ran
(5.23.0), so the tokenizer and the pipeline behave the same way.

**Look for:** no error. pip may print a line about restarting; you do not need to restart.

In [1]:
%pip install -q transformers==5.23.0

Note: you may need to restart the kernel to use updated packages.


## 2. The six notes and their tags

These are the six notes from the lesson, split on spaces, with the tags a person gave them. **B-** marks the
first word of a supplier, a date or an amount, **I-** each further word of the same one, **O** every other word.
`TAG_IDS` turns a tag into the number the model learns (O is 0, B-SUPPLIER is 1, and so on).

**Look for:** note 4. "Busan" is B-SUPPLIER although the word before it ("Textiles") is also a supplier word:
the B- tag is what keeps the two suppliers apart.

In [2]:
TAGS = ["O", "B-SUPPLIER", "I-SUPPLIER", "B-DATE", "I-DATE", "B-AMOUNT", "I-AMOUNT"]
TAG_IDS = {t: i for i, t in enumerate(TAGS)}

NOTES = [
    ("Paid Han Textiles 1,250,000 won on 3 March for linen",
     "O B-SUPPLIER I-SUPPLIER B-AMOUNT I-AMOUNT O B-DATE I-DATE O O"),
    ("Deposit to Busan Wire Co 400 USD on 12 March",
     "O O B-SUPPLIER I-SUPPLIER I-SUPPLIER B-AMOUNT I-AMOUNT O B-DATE I-DATE"),
    ("Refund from Kim Trading 85,000 won 2 April",
     "O O B-SUPPLIER I-SUPPLIER B-AMOUNT I-AMOUNT B-DATE I-DATE"),
    ("Split payment Han Textiles Busan Wire 600 USD each on 9 April",
     "O O B-SUPPLIER I-SUPPLIER B-SUPPLIER I-SUPPLIER B-AMOUNT I-AMOUNT O O B-DATE I-DATE"),
    ("Second half to Mekong Ceramics 2,300 USD sent 14 March",
     "O O O B-SUPPLIER I-SUPPLIER B-AMOUNT I-AMOUNT O B-DATE I-DATE"),
    ("Samarkand Silk late fee 15 USD due 30 April",
     "B-SUPPLIER I-SUPPLIER O O B-AMOUNT I-AMOUNT O B-DATE I-DATE"),
]
NOTES = [(text.split(), tags.split()) for text, tags in NOTES]
for words, tags in NOTES:
    assert len(words) == len(tags), words

for n, (words, tags) in enumerate(NOTES, start=1):
    print(f"note {n}:", "  ".join(f"{w}/{t}" for w, t in zip(words, tags)))

note 1: Paid/O  Han/B-SUPPLIER  Textiles/I-SUPPLIER  1,250,000/B-AMOUNT  won/I-AMOUNT  on/O  3/B-DATE  March/I-DATE  for/O  linen/O
note 2: Deposit/O  to/O  Busan/B-SUPPLIER  Wire/I-SUPPLIER  Co/I-SUPPLIER  400/B-AMOUNT  USD/I-AMOUNT  on/O  12/B-DATE  March/I-DATE
note 3: Refund/O  from/O  Kim/B-SUPPLIER  Trading/I-SUPPLIER  85,000/B-AMOUNT  won/I-AMOUNT  2/B-DATE  April/I-DATE
note 4: Split/O  payment/O  Han/B-SUPPLIER  Textiles/I-SUPPLIER  Busan/B-SUPPLIER  Wire/I-SUPPLIER  600/B-AMOUNT  USD/I-AMOUNT  each/O  on/O  9/B-DATE  April/I-DATE
note 5: Second/O  half/O  to/O  Mekong/B-SUPPLIER  Ceramics/I-SUPPLIER  2,300/B-AMOUNT  USD/I-AMOUNT  sent/O  14/B-DATE  March/I-DATE
note 6: Samarkand/B-SUPPLIER  Silk/I-SUPPLIER  late/O  fee/O  15/B-AMOUNT  USD/I-AMOUNT  due/O  30/B-DATE  April/I-DATE


## 3. The tokenizer cuts words into pieces

The model does not see your words. Its tokenizer lowercases them, cuts them into pieces from its vocabulary
(a piece that continues a word starts with `##`) and adds `[CLS]` at the start and `[SEP]` at the end.
`is_split_into_words=True` says the note is already a list of words, so `word_ids()` can tell, for each
token, which word it came from (`None` for the added tokens).

**Look for:** in note 3, "Refund" becomes `ref` + `##und` (both word 0), and "85,000" becomes three tokens
`85`, `,`, `000` (all word 4). 8 words give 13 tokens.

In [3]:
from transformers import AutoTokenizer

MODEL_NAME = "google/bert_uncased_L-4_H-256_A-4"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

words, tags = NOTES[2]
encoding = tokenizer(words, is_split_into_words=True)
print("note 3:", len(words), "words ->", len(encoding.tokens()), "tokens")
print(f"{'token':>10}  word_id  word")
for token, word_id in zip(encoding.tokens(), encoding.word_ids()):
    print(f"{token:>10}  {str(word_id):>7}  {'' if word_id is None else words[word_id]}")

note 3: 8 words -> 13 tokens
     token  word_id  word
     [CLS]     None  
       ref        0  Refund
     ##und        0  Refund
      from        1  from
       kim        2  Kim
   trading        3  Trading
        85        4  85,000
         ,        4  85,000
       000        4  85,000
       won        5  won
         2        6  2
     april        7  April
     [SEP]     None  


## 4. Align the word tags to the tokens

This is the function from the lesson's code screen. The first token of a word gets the word's tag. Later
pieces of the same word, and the added tokens, get **-100**: PyTorch's cross-entropy loss skips targets equal to
-100 (its `ignore_index`), so each word counts once in training.

**Look for:** note 2 gives 13 labels, and 3 of them are -100: `[CLS]`, the piece `##an` and `[SEP]`.

In [4]:
def align_labels(note_words, word_tags):
    encoding = tokenizer(note_words, is_split_into_words=True)
    labels = []
    previous = None
    for word_id in encoding.word_ids():
        if word_id is None or word_id == previous:
            labels.append(-100)
        else:
            labels.append(TAG_IDS[word_tags[word_id]])
        previous = word_id
    return labels


def label_name(i):
    return "-100" if i == -100 else TAGS[i]


words, tags = NOTES[1]
labels = align_labels(words, tags)
tokens = tokenizer(words, is_split_into_words=True).tokens()
print("note 2:", " ".join(words))
for token, label in zip(tokens, labels):
    print(f"{token:>10}  {label_name(label)}")
print("labels:", len(labels), "| of them -100:", labels.count(-100))

note 2: Deposit to Busan Wire Co 400 USD on 12 March
     [CLS]  -100
   deposit  O
        to  O
       bus  B-SUPPLIER
      ##an  -100
      wire  I-SUPPLIER
        co  I-SUPPLIER
       400  B-AMOUNT
       usd  I-AMOUNT
        on  O
        12  B-DATE
     march  I-DATE
     [SEP]  -100
labels: 13 | of them -100: 3


## 5. The broken alignment: labels written by position

A colleague's version copies the word tags onto token positions, as if words and tokens were the same list, and
pads the rest with -100. The next cell prints both versions for note 4 next to each other.

**Look for:** the first token where the two columns differ is `[CLS]`: it gets the first word's tag, O. From
there every label sits one token too early (`payment` holds the tag meant for `han`), and after the split word
`bus ##an` the labels slip one token more.

In [5]:
def align_labels_by_position(note_words, word_tags):
    encoding = tokenizer(note_words, is_split_into_words=True)
    labels = [TAG_IDS[tag] for tag in word_tags]
    labels += [-100] * (len(encoding["input_ids"]) - len(labels))
    return labels


words, tags = NOTES[3]
good = align_labels(words, tags)
bad = align_labels_by_position(words, tags)
tokens = tokenizer(words, is_split_into_words=True).tokens()
print(f"{'token':>10}  {'word_ids':<12}  {'by position':<12}")
for token, g, b in zip(tokens, good, bad):
    mark = "" if g == b else "  <- differs"
    print(f"{token:>10}  {label_name(g):<12}  {label_name(b):<12}{mark}")
first = next(i for i, (g, b) in enumerate(zip(good, bad)) if g != b)
print("first differing token:", tokens[first])

     token  word_ids      by position 
     [CLS]  -100          O             <- differs
     split  O             O           
   payment  O             B-SUPPLIER    <- differs
       han  B-SUPPLIER    I-SUPPLIER    <- differs
  textiles  I-SUPPLIER    B-SUPPLIER    <- differs
       bus  B-SUPPLIER    I-SUPPLIER    <- differs
      ##an  -100          B-AMOUNT      <- differs
      wire  I-SUPPLIER    I-AMOUNT      <- differs
       600  B-AMOUNT      O             <- differs
       usd  I-AMOUNT      O             <- differs
      each  O             B-DATE        <- differs
        on  O             I-DATE        <- differs
         9  B-DATE        -100          <- differs
     april  I-DATE        -100          <- differs
     [SEP]  -100          -100        
first differing token: [CLS]


## 6. Generate 600 training notes

Six notes are far too few to train on. The lesson's run generated 600 notes from templates worded like the six,
with other supplier names (none of them appears in the six notes), so the model is tested on names it never saw.
The generator is seeded, so you get the same 600 notes as the lesson.

**Look for:** 600 notes, and the first one is "Navoi Copper late fee 66,000 won due 16 June".

In [6]:
import random

SUPPLIERS = ["Lotus Paper", "Orient Bolts", "Daegu Cotton Mill", "Andijan Fruit", "Pacific Glassworks", "Nam Electric",
             "Fergana Silk House", "Seoul Fasteners", "Bukhara Carpets", "Delta Packaging", "Jeju Tea Farm",
             "Golden Thread Co", "Incheon Tools", "Navoi Copper", "Green Valley Foods", "Ulsan Steel Works",
             "Khiva Pottery", "Blue Harbor Freight", "Gwangju Plastics", "Silver Leaf Trading"]
MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October",
          "November", "December"]
CURRENCIES = ["USD", "won", "EUR"]
TEMPLATES = [
    "Paid {S} {A} on {D} for {X}",
    "Payment to {S} {A} {D}",
    "Deposit to {S} {A} on {D}",
    "{S} invoice {A} paid {D}",
    "Refund from {S} {A} {D}",
    "Second half to {S} {A} sent {D}",
    "{S} late fee {A} due {D}",
    "Transfer {A} to {S} on {D} for {X}",
    "On {D} sent {A} to {S}",
    "Split payment {S} {S2} {A} each on {D}",
    "Advance for {X} to {S} {A} {D}",
    "{D} paid {S} {A} balance",
]
GOODS = ["linen", "buttons", "zippers", "glass jars", "boxes", "dried apricots", "copper wire", "tea", "rope", "paper"]


def amount(rng):
    n = rng.choice([rng.randint(5, 999), rng.randint(1, 99) * 1000, rng.randint(1, 3000) * 1000])
    return [f"{n:,}", rng.choice(CURRENCIES)]


def span(words, kind):
    return list(zip(words, [f"B-{kind}"] + [f"I-{kind}"] * (len(words) - 1)))


def make_training_notes(n, seed=0):
    rng = random.Random(seed)
    notes = []
    for _ in range(n):
        tpl = rng.choice(TEMPLATES)
        s, s2 = rng.sample(SUPPLIERS, 2)
        parts = {
            "{S}": span(s.split(), "SUPPLIER"),
            "{S2}": span(s2.split(), "SUPPLIER"),
            "{A}": span(amount(rng), "AMOUNT"),
            "{D}": span([str(rng.randint(1, 28)), rng.choice(MONTHS)], "DATE"),
            "{X}": [(w, "O") for w in rng.choice(GOODS).split()],
        }
        pairs = []
        for piece in tpl.split():
            pairs += parts.get(piece, [(piece, "O")])
        notes.append(([w for w, _ in pairs], [t for _, t in pairs]))
    return notes


train_notes = make_training_notes(600, seed=0)
print("training notes:", len(train_notes))
for words, tags in train_notes[:3]:
    print("  ".join(f"{w}/{t}" for w, t in zip(words, tags)))

training notes: 600
Navoi/B-SUPPLIER  Copper/I-SUPPLIER  late/O  fee/O  66,000/B-AMOUNT  won/I-AMOUNT  due/O  16/B-DATE  June/I-DATE
Khiva/B-SUPPLIER  Pottery/I-SUPPLIER  invoice/O  389,000/B-AMOUNT  won/I-AMOUNT  paid/O  18/B-DATE  December/I-DATE
Deposit/O  to/O  Delta/B-SUPPLIER  Packaging/I-SUPPLIER  10,000/B-AMOUNT  won/I-AMOUNT  on/O  18/B-DATE  February/I-DATE


## 7. Fine-tune the model with the word_ids alignment

`train()` is the lesson's training script as a function: it aligns the labels of every training note with the
function you pass in, loads the small BERT with a fresh 7-label head, and trains for 4 epochs with AdamW,
16 notes per batch, on the CPU. Seeds are fixed, so the run repeats. It takes about half a minute. The cell
also hides the long load report transformers prints when it adds a new, untrained head to a model.

**Look for:** the mean training loss after epoch 1 (the lesson's run: 0.3492) is far above the loss after
epoch 2 (0.0222). Your numbers may differ a little on other hardware.

In [7]:
import time
import torch
from transformers import AutoModelForTokenClassification
from transformers.utils import logging

logging.set_verbosity_error()  # hide the long load report about the new 7-label head
logging.disable_progress_bar()


def train(align, n_notes=600, epochs=4):
    torch.manual_seed(0)
    t0 = time.time()
    rows = []
    for words, tags in make_training_notes(n_notes, seed=0):
        enc = tokenizer(words, is_split_into_words=True)
        rows.append((enc["input_ids"], align(words, tags)))
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME, num_labels=len(TAGS), id2label=dict(enumerate(TAGS)), label2id=TAG_IDS)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    BATCH = 16
    losses = []
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(rows)).tolist()
        total, batches = 0.0, 0
        for b in range(0, len(order), BATCH):
            chunk = [rows[i] for i in order[b:b + BATCH]]
            width = max(len(ids) for ids, _ in chunk)
            ids = torch.tensor([x + [0] * (width - len(x)) for x, _ in chunk])
            mask = torch.tensor([[1] * len(x) + [0] * (width - len(x)) for x, _ in chunk])
            labels = torch.tensor([y + [-100] * (width - len(y)) for _, y in chunk])
            loss = model(input_ids=ids, attention_mask=mask, labels=labels).loss
            loss.backward(); opt.step(); opt.zero_grad()
            total += loss.item(); batches += 1
        losses.append(total / batches)
        print(f"epoch {epoch + 1} mean training loss {losses[-1]:.4f}")
    model.eval()
    print(f"trained in {time.time() - t0:.0f} seconds")
    return model, losses


model, losses = train(align_labels)

epoch 1 mean training loss 0.3492


epoch 2 mean training loss 0.0222


epoch 3 mean training loss 0.0142


epoch 4 mean training loss 0.0179
trained in 20 seconds


## 8. Train again with the broken alignment

Same model, same 600 notes, same seeds. Only the alignment function changes.

**Look for:** the loss of the last epoch stays well above the word_ids version (the lesson's run: 0.1390
against 0.0179). The model is learning the shifted labels, which are harder to fit.

In [8]:
buggy_model, buggy_losses = train(align_labels_by_position)
print()
print("last epoch loss, word_ids alignment:   ", round(losses[-1], 4))
print("last epoch loss, by-position alignment:", round(buggy_losses[-1], 4))

epoch 1 mean training loss 1.0101


epoch 2 mean training loss 0.3905


epoch 3 mean training loss 0.2538


epoch 4 mean training loss 0.1390
trained in 13 seconds

last epoch loss, word_ids alignment:    0.0179
last epoch loss, by-position alignment: 0.139


## 9. Run the tagger, token by token

Hugging Face's pipeline turns the model's output into a list of entities. With `aggregation_strategy="none"`
it returns one row per token that the model did not label O. The score is the model's probability for that label.
`show()` prints the rows and nothing else.

**Look for:** in note 5, the piece `##kong` has its own row, and it says B-SUPPLIER, not I-SUPPLIER. In training
every later piece had -100, so no example ever told the model what to say for it.

In [9]:
from transformers import pipeline


def show(entities):
    for e in entities:
        label = e.get("entity_group", e.get("entity"))
        print(f"  {label:<11} {e['word']:<18} {float(e['score']):.3f}")


note = " ".join(NOTES[4][0])
tagger = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="none")
print("note 5:", note)
show(tagger(note))

note 5: Second half to Mekong Ceramics 2,300 USD sent 14 March


  B-SUPPLIER  me                 0.993
  B-SUPPLIER  ##kong             0.656
  I-SUPPLIER  ceramics           0.991
  B-AMOUNT    2                  0.994
  B-AMOUNT    ,                  0.989
  B-AMOUNT    300                0.993
  I-AMOUNT    usd                0.993
  B-DATE      14                 0.994
  I-DATE      march              0.993


## 10. Join the pieces: "simple" against "first"

`"simple"` joins tokens into entities: a B- tag starts one, and I- tags of the same kind right after it join it.
`"first"` works on words: every word takes the label of its first token, the one the model was trained on.

**Look for:** with "simple" the supplier of note 5 comes out as "me" and "##kong ceramics". With "first" it is
"mekong ceramics", and "samarkand silk" in note 6 is whole too. The amount "2,300 usd" is still in three parts with
"first": the comma and "300" have no ##, so the pipeline counts each as a word of its own.

In [10]:
for strategy in ["simple", "first"]:
    tagger = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy=strategy)
    print(f'aggregation_strategy="{strategy}"')
    found = []
    for words, _ in NOTES[3:]:
        note = " ".join(words)
        entities = tagger(note)
        found += [e["word"] for e in entities]
        print(" ", note)
        show(entities)
    print("  words that start with ##:", [w for w in found if w.startswith("##")])
    print()

aggregation_strategy="simple"
  Split payment Han Textiles Busan Wire 600 USD each on 9 April
  SUPPLIER    han textiles       0.992
  SUPPLIER    busan wire         0.848
  AMOUNT      600 usd            0.992
  DATE        9 april            0.994
  Second half to Mekong Ceramics 2,300 USD sent 14 March
  SUPPLIER    me                 0.993
  SUPPLIER    ##kong ceramics    0.823
  AMOUNT      2                  0.994
  AMOUNT      ,                  0.989
  AMOUNT      300 usd            0.993
  DATE        14 march           0.993


  Samarkand Silk late fee 15 USD due 30 April
  SUPPLIER    sam                0.986
  SUPPLIER    ##arkand silk      0.980
  AMOUNT      15 usd             0.872
  DATE        30 april           0.994
  words that start with ##: ['##kong ceramics', '##arkand silk']

aggregation_strategy="first"
  Split payment Han Textiles Busan Wire 600 USD each on 9 April
  SUPPLIER    han textiles       0.992
  SUPPLIER    busan wire         0.871
  AMOUNT      600 usd            0.992
  DATE        9 april            0.994
  Second half to Mekong Ceramics 2,300 USD sent 14 March
  SUPPLIER    mekong ceramics    0.992
  AMOUNT      2                  0.994
  AMOUNT      ,                  0.989
  AMOUNT      300 usd            0.993
  DATE        14 march           0.993
  Samarkand Silk late fee 15 USD due 30 April
  SUPPLIER    samarkand silk     0.990
  AMOUNT      15 usd             0.872
  DATE        30 april           0.994
  words that start with ##: []



## 11. The broken model, on the same notes

The model trained with the by-position labels, run with `"first"`.

**Look for:** in note 4 the suppliers are "payment han" and "textiles busan": every entity starts one word too
early, exactly like the shifted labels you saw in step 5.

In [11]:
tagger = pipeline("token-classification", model=buggy_model, tokenizer=tokenizer, aggregation_strategy="first")
for words, _ in NOTES[3:]:
    note = " ".join(words)
    print(note)
    show(tagger(note))

Split payment Han Textiles Busan Wire 600 USD each on 9 April
  SUPPLIER    payment han        0.996
  SUPPLIER    textiles busan     0.981
  AMOUNT      wire               0.611
  DATE        each on 9          0.711
Second half to Mekong Ceramics 2,300 USD sent 14 March
  SUPPLIER    to mekong          0.994
  AMOUNT      ceramics           0.934
  DATE        , 300 usd sent 14 march 0.922
Samarkand Silk late fee 15 USD due 30 April
  SUPPLIER    samarkand          0.996
  AMOUNT      late               0.589
  DATE        15 usd due 30 april 0.853


## Now change it

Each cell below has one line marked `# <- change this`. Edit it and run the cell again.

### Experiment 1: a note the model has never seen

**What should change:** write your own transfer note in the same style, with a new supplier name. With
`"first"` each word gets one label, so a new name comes out whole as long as the note looks like the training
notes. As given, "tashkent leather" comes out whole, and the amount splits at its comma again ("7", ",",
"500 won"). Try a wording no template uses (for example "Sent to Tashkent Leather ..." or a date like "21/06") and
watch where the tags go wrong: the model only knows the patterns in its training notes.

In [12]:
note = "Transfer 7,500 won to Tashkent Leather on 21 June for belts"  # <- change this
tagger = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="first")
print(note)
show(tagger(note))

Transfer 7,500 won to Tashkent Leather on 21 June for belts
  AMOUNT      7                  0.991
  AMOUNT      ,                  0.989
  AMOUNT      500 won            0.991
  SUPPLIER    tashkent leather   0.993
  DATE        21 june            0.994


### Experiment 2: the other word-level modes

**What should change:** `"average"` and `"max"` also work on words, so "samarkand silk" stays whole, like with
`"first"`; only the scores change a little. Put back `"simple"` and the name splits into "sam" and "##arkand silk"
again, because "simple" reads the untrained later pieces too.

In [13]:
strategy = "average"  # <- change this (try "first", "max" or "simple")
tagger = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy=strategy)
note = " ".join(NOTES[5][0])
print(f'aggregation_strategy="{strategy}":', note)
show(tagger(note))

aggregation_strategy="average": Samarkand Silk late fee 15 USD due 30 April
  SUPPLIER    samarkand silk     0.822
  AMOUNT      15 usd             0.872
  DATE        30 april           0.994


### Experiment 3: fewer training notes

The cell retrains the word_ids model on fewer generated notes and runs it on note 5.

**What should change:** with 60 notes there are only 4 batches per epoch. The loss after 4 epochs is about
ten times higher than with 600 notes (about 0.18 against 0.018), and the scores on note 5 drop from about 0.99 to
between 0.7 and 0.93. Raise the number back towards 600 and the loss falls again. Tagged
examples are what the model learns from; that tagging time is the real cost of a token classifier.

In [14]:
n_notes = 60  # <- change this (try 150 or 600)
small_model, small_losses = train(align_labels, n_notes=n_notes)
tagger = pipeline("token-classification", model=small_model, tokenizer=tokenizer, aggregation_strategy="first")
note = " ".join(NOTES[4][0])
print(note)
show(tagger(note))

epoch 1 mean training loss 1.5912
epoch 2 mean training loss 0.7984


epoch 3 mean training loss 0.3858
epoch 4 mean training loss 0.1787
trained in 1 seconds


Second half to Mekong Ceramics 2,300 USD sent 14 March
  SUPPLIER    mekong ceramics    0.930
  AMOUNT      2,                 0.701
  AMOUNT      300 usd            0.730
  DATE        14 march           0.917


## What you should see

- Step 4: `align_labels` on "Deposit to Busan Wire Co 400 USD on 12 March" returns 13 labels, 3 of them -100.
- Step 7: the mean training loss after epoch 1 (the lesson's run: 0.3492) is far above the loss after epoch 2
  (0.0222).
- Step 8: trained with `align_labels_by_position` instead, the last epoch's loss stays well above the word_ids
  version (the lesson's run: 0.139 against 0.0179).
- Step 9: with `aggregation_strategy="none"` a token whose text starts with ## (`##kong`) appears in the output,
  with its own label.
- Step 10: with `aggregation_strategy="first"` no returned word starts with ## (the cell prints an empty list).